# 01 · Identity and onboarding (Block A)

A freshly deployed dataspace is empty of trust: nobody holds a credential, so every policy fails.
This notebook makes the authority issue credentials to every participant.

## Concept: Verifiable Credentials

A **Verifiable Credential (VC)** is a set of claims about a subject, signed by an issuer. It follows the W3C VC data model:

```
issuer  (authority DID)   ──signs──►   credential   ──held by──►   holder (participant wallet)
                                       "consumer is a FullMember,
                                        domain = route"
verifier (another connector) checks:   the signature (issuer's public key, from its DID document)
                                       the issuer is trusted
                                       not expired, not revoked
```

Three roles:

| Role | Here |
|---|---|
| **Issuer** | `authority-issuerservice`, signing with the authority's key |
| **Holder** | each participant's identity hub (wallet) |
| **Verifier** | the provider's control plane, when the consumer asks for its catalog or a contract |

In this dataspace the credentials are **JWTs** (`VC1_0_JWT`): a signed token whose payload contains the credential.
Every connector is configured with the authority's DID as its **only trusted issuer**.

The credential types used here:

| Type | Claims | Used for |
|---|---|---|
| `MembershipCredential` | name, membership type, since, company segment | every policy requires an active membership |
| `DomainCredential` | domain (`route`) | policies restricted to a business domain |
| `VisibilityCredential` | visibility scope (`ALL`) | lets its holder see restricted assets in catalogs (given by the E2E suite) |

## Concept: DCP, the Decentralized Claims Protocol

DCP standardises two exchanges between wallets and the outside world:

1. **Issuance**: the holder's wallet sends a `CredentialRequestMessage` to the issuer's `IssuerService` endpoint, authenticated with a token signed by the holder.
   The issuer checks its records, signs the credential and sends a `CredentialMessage` to the holder's `CredentialService`.
2. **Presentation**: a verifier receives a token from the holder and queries the holder's `CredentialService` for a *Verifiable Presentation* of the credentials it needs.
   This happens on every DSP call (notebooks 03–05).

The tokens are minted by each wallet's **STS** (Secure Token Service) and signed with the participant's private key.

## Concept: how the issuer decides what to put in a credential

The issuer service never invents claims. For each credential type it has:

- an **attestation definition**: a source of facts about holders, such as a database table;
- a **credential definition**: the credential type to issue, which attestations feed it, how fields map to claims (`mappings`), the format and the validity;
- **holders**: the participants it knows, by DID;
- **attestations**: the actual facts per holder (rows in those tables), written through the DSE admin APIs.

When a request arrives, the issuer runs the attestations for that holder, maps the results to claims, signs, and records an **issuance process**.

In [1]:
import dataspace as ds

## A1 · Define the `MembershipCredential` type

### Attestation definition

**What:** `POST /authority/is/issueradmin/.../attestations`. Registers a source of facts.
`json-database` means: read the row for the holder from table `membership_attestation`, plus the JSON column `properties` (for `companySegment`).

**Returns:** `201 Created` with an empty body, or `409 Conflict` if it already exists (fine on a re-run).

In [2]:
membership_attestation = {
    "id": "membership-attestation-def-1",
    "attestationType": "json-database",
    "configuration": {
        "dataSourceName": "membership",
        "tableName": "membership_attestation",
        "propertiesColumn": "properties",
    },
}
ds.post(f"{ds.issuer_admin_api()}/attestations", membership_attestation, expect={201, 409})

POST   /authority/is/issueradmin/v1alpha/participants/ZGlkOndlYjphdXRob3JpdHktaWRlbnRpdHlodWIlM0E4MzgzOmFwaTpkaWQ=/attestations  ->  409


<Response [409]>

### Credential definition

**What:** `POST .../credentialdefinitions`. Says "a `MembershipCredential` is built from `membership-attestation-def-1`, with these claims, as a JWT valid for 10 years".

`mappings` connects each attestation field (`input`) to a JSON path in the credential (`output`). With `required: true`, issuance fails if the fact is missing.

**Returns:** 201 (or 409).

In [3]:
membership_credential = {
    "id": "membership-credential-def-1",
    "credentialType": "MembershipCredential",
    "attestations": ["membership-attestation-def-1"],
    "format": "VC1_0_JWT",
    "validity": 10 * 365 * 24 * 3600,   # 10 years, in seconds
    "jsonSchemaUrl": "https://example.com/schema/membership-credential.json",
    "mappings": [
        # attestation field     -> claim inside the credential
        {"input": "name",                  "output": "credentialSubject.name",                     "required": True},
        {"input": "membership_type",       "output": "credentialSubject.membership.membershipType", "required": True},
        {"input": "membership_start_date", "output": "credentialSubject.membership.since",          "required": True},
        {"input": "companySegment",        "output": "credentialSubject.companySegment",            "required": False},
    ],
}
ds.post(f"{ds.issuer_admin_api()}/credentialdefinitions", membership_credential, expect={201, 409})

POST   /authority/is/issueradmin/v1alpha/participants/ZGlkOndlYjphdXRob3JpdHktaWRlbnRpdHlodWIlM0E4MzgzOmFwaTpkaWQ=/credentialdefinitions  ->  409


<Response [409]>

## A1 · Define the `DomainCredential` type

Same two steps. `database` attestations read plain columns, here `domain` from table `domain_attestation`.

In [4]:
domain_attestation = {
    "id": "domain-attestation-def-1",
    "attestationType": "database",
    "configuration": {"dataSourceName": "domain", "tableName": "domain_attestation"},
}
ds.post(f"{ds.issuer_admin_api()}/attestations", domain_attestation, expect={201, 409})

domain_credential = {
    "id": "domain-credential-def-1",
    "credentialType": "DomainCredential",
    "attestations": ["domain-attestation-def-1"],
    "format": "VC1_0_JWT",
    "validity": 10 * 365 * 24 * 3600,
    "jsonSchemaUrl": "https://example.com/schema/domain-credential.json",
    "mappings": [{"input": "domain", "output": "credentialSubject.domain", "required": True}],
}
ds.post(f"{ds.issuer_admin_api()}/credentialdefinitions", domain_credential, expect={201, 409})

POST   /authority/is/issueradmin/v1alpha/participants/ZGlkOndlYjphdXRob3JpdHktaWRlbnRpdHlodWIlM0E4MzgzOmFwaTpkaWQ=/attestations  ->  409
POST   /authority/is/issueradmin/v1alpha/participants/ZGlkOndlYjphdXRob3JpdHktaWRlbnRpdHlodWIlM0E4MzgzOmFwaTpkaWQ=/credentialdefinitions  ->  409


<Response [409]>

### What the issuer has now

**What:** `POST .../credentialdefinitions/query` with an empty query (`{}`) returns every definition.
Endpoints ending in `/query` or `/request` are EDC's convention for "search", taking a filter in the body.

**Returns:** a JSON array of definitions as stored, with the defaults the issuer filled in (`rules`, `participantContextId`…).

In [5]:
definitions = ds.post(f"{ds.issuer_admin_api()}/credentialdefinitions/query", {}).json()
for definition in definitions:
    print(f"   {definition['id']:32} issues {definition['credentialType']}")

print("\nOne definition as stored:")
ds.show(definitions[0], max_chars=1200)

POST   /authority/is/issueradmin/v1alpha/participants/ZGlkOndlYjphdXRob3JpdHktaWRlbnRpdHlodWIlM0E4MzgzOmFwaTpkaWQ=/credentialdefinitions/query  ->  200
   membership-credential-def-1      issues MembershipCredential
   domain-credential-def-1          issues DomainCredential
   visibility-credential-def-1      issues VisibilityCredential

One definition as stored:
{
  "id": "membership-credential-def-1",
  "createdAt": 1791549462374,
  "participantContextId": "did:web:authority-identityhub%3A8383:api:did",
  "attestations": [
    "membership-attestation-def-1"
  ],
  "rules": [],
  "mappings": [
    {
      "input": "name",
      "output": "credentialSubject.name",
      "required": true
    },
    {
      "input": "membership_type",
      "output": "credentialSubject.membership.membershipType",
      "required": true
    },
    {
      "input": "membership_start_date",
      "output": "credentialSubject.membership.since",
      "required": true
    },
    {
      "input": "companySegm

## A2 · Register the participants

For each participant the authority writes three records:

| Call | Stores | Returns |
|---|---|---|
| `POST .../holders` | the participant as a known **holder** (DID + name). The federated catalog also crawls exactly this list | 201 / 409 |
| `POST .../attestation-membership` (DSE API) | a row in `membership_attestation`: name, `FullMember`, today's date, `companySegment` | 204 / 409 |
| `POST .../attestation-domain` (DSE API) | a row in `domain_attestation`: domain `route` | 204 |

The `participant_context` in the URL (`ds.issuer_admin_api()`) is the **authority's** context: the issuer is multi-tenant, and these records belong to the authority as issuer.

In [6]:
for participant in ds.PARTICIPANTS:
    participant_did = ds.did(participant)
    print(f"-- {participant}")

    holder = {"did": participant_did, "holderId": participant_did, "name": participant, "properties": {}}
    ds.post(f"{ds.issuer_admin_api()}/holders", holder, expect={201, 409})

    membership = {
        "id": participant_did,
        "holderId": participant_did,
        "name": participant,
        "membershipType": "FullMember",
        "properties": {"companySegment": "Airlines"},
    }
    ds.post(f"{ds.issuer_admin_api()}/attestation-membership", membership, expect={204, 409})

    domain = {"holderId": participant_did, "domain": "route"}
    ds.post(f"{ds.issuer_admin_api()}/attestation-domain", domain, expect={204})

-- authority
POST   /authority/is/issueradmin/v1alpha/participants/ZGlkOndlYjphdXRob3JpdHktaWRlbnRpdHlodWIlM0E4MzgzOmFwaTpkaWQ=/holders  ->  409
POST   /authority/is/issueradmin/v1alpha/participants/ZGlkOndlYjphdXRob3JpdHktaWRlbnRpdHlodWIlM0E4MzgzOmFwaTpkaWQ=/attestation-membership  ->  409
POST   /authority/is/issueradmin/v1alpha/participants/ZGlkOndlYjphdXRob3JpdHktaWRlbnRpdHlodWIlM0E4MzgzOmFwaTpkaWQ=/attestation-domain  ->  204
-- provider
POST   /authority/is/issueradmin/v1alpha/participants/ZGlkOndlYjphdXRob3JpdHktaWRlbnRpdHlodWIlM0E4MzgzOmFwaTpkaWQ=/holders  ->  409
POST   /authority/is/issueradmin/v1alpha/participants/ZGlkOndlYjphdXRob3JpdHktaWRlbnRpdHlodWIlM0E4MzgzOmFwaTpkaWQ=/attestation-membership  ->  409
POST   /authority/is/issueradmin/v1alpha/participants/ZGlkOndlYjphdXRob3JpdHktaWRlbnRpdHlodWIlM0E4MzgzOmFwaTpkaWQ=/attestation-domain  ->  204
-- consumer
POST   /authority/is/issueradmin/v1alpha/participants/ZGlkOndlYjphdXRob3JpdHktaWRlbnRpdHlodWIlM0E4MzgzOmFwaTpkaWQ=/hold

## A3 · This issuer can only attest the `route` domain

**What:** try to record `domain = travel` for the consumer.

**Why it fails:** Terraform deploys the issuer with `domain: "route"`. It is the authority *for that domain only*, and its domain API refuses any other value.

**Returns:** **403 Forbidden**. That is why nobody can hold a `travel` DomainCredential, which notebook 02 uses on purpose.

In [7]:
travel = {"holderId": ds.did("consumer"), "domain": "travel"}
ds.post(f"{ds.issuer_admin_api()}/attestation-domain", travel, expect={403})

POST   /authority/is/issueradmin/v1alpha/participants/ZGlkOndlYjphdXRob3JpdHktaWRlbnRpdHlodWIlM0E4MzgzOmFwaTpkaWQ=/attestation-domain  ->  403


<Response [403]>

## A4 · Each participant obtains its credentials

**What:** `POST /<p>/ih/identity/v1alpha/participants/<context>/credentials/request` on the participant's **own** wallet.

**Why the wallet and not the issuer:** in DCP the *holder* starts issuance. The wallet resolves the issuer's DID, finds its `IssuerService`
endpoint, and sends the request signed with the participant's key, so the issuer knows who is asking.

| Field | Meaning |
|---|---|
| `issuerDid` | whom to ask |
| `holderPid` | an id *we* choose for this request, to track it |
| `credentials` | list of `{format, type, id}`: format, credential type, credential definition id |

The descriptor field is **`type`**. With `credentialType` instead, the issuer approves the request but never delivers the credential.

**Returns:** `201` immediately. Issuance then continues asynchronously. Credentials a wallet already holds are not requested again.

In [8]:
CREDENTIALS_TO_REQUEST = {
    "MembershipCredential": "membership-credential-def-1",
    "DomainCredential": "domain-credential-def-1",
}
run_id = ds.recall("run_id", produced_by="00_environment.ipynb")
requests_sent = []


def credential_types_in_wallet(participant):
    stored = ds.get(f"{ds.identity_api(participant)}/credentials", quiet=True).json()
    types = set()
    for entry in stored:
        types.update(entry["verifiableCredential"]["credential"]["type"])
    return types


for participant in ds.PARTICIPANTS:
    already_held = credential_types_in_wallet(participant)
    for credential_type, definition_id in CREDENTIALS_TO_REQUEST.items():
        if credential_type in already_held:
            print(f"   {participant:10} already holds {credential_type}")
            continue
        holder_pid = f"{participant}-{credential_type}-{run_id}"
        credential_request = {
            "issuerDid": ds.did("authority"),
            "holderPid": holder_pid,
            "credentials": [{"format": "VC1_0_JWT", "type": credential_type, "id": definition_id}],
        }
        ds.post(f"{ds.identity_api(participant)}/credentials/request", credential_request, expect={201})
        requests_sent.append((participant, holder_pid))

   authority  already holds MembershipCredential
   authority  already holds DomainCredential
   provider   already holds MembershipCredential
   provider   already holds DomainCredential
   consumer   already holds MembershipCredential
   consumer   already holds DomainCredential


Wait until every wallet holds both credentials (normally about a second).

In [9]:
for participant in ds.PARTICIPANTS:
    def wallet_is_complete():
        return set(CREDENTIALS_TO_REQUEST) <= credential_types_in_wallet(participant)

    ds.wait_until(f"{participant} holds Membership + Domain", wallet_is_complete)

   authority holds Membership + Domain: yes, after 0s
   provider holds Membership + Domain: yes, after 0s
   consumer holds Membership + Domain: yes, after 0s


### Following a request from the wallet's side

**What:** `GET .../credentials/request/<holderPid>`.  **Returns:** the wallet's record of the request.
`status` goes `CREATED → REQUESTING → REQUESTED → ISSUED`, and `issuerPid` is the issuer's id for the same exchange.
Nothing is shown if every wallet already held its credentials.

In [10]:
for participant, holder_pid in requests_sent:
    request_record = ds.get(f"{ds.identity_api(participant)}/credentials/request/{holder_pid}").json()
    print(f"   {holder_pid:40} status={request_record['status']}  issuerPid={request_record['issuerPid']}")
if not requests_sent:
    print("   (no new requests in this run)")

   (no new requests in this run)


## A5 · Anatomy of a stored credential

**What:** `GET .../credentials` on the consumer's wallet.  **Returns:** an array of *credential resources*. Each has:

| Field | Meaning |
|---|---|
| `state` | wallet-side lifecycle state, as a numeric code: 500 = `ISSUED` (usable), 600 = `REVOKED`, 700 = `SUSPENDED`, 800 = `EXPIRED` |
| `issuerId`, `holderId` | DIDs |
| `verifiableCredential.rawVc` | **the JWT exactly as signed by the issuer**: what the wallet presents to verifiers |
| `verifiableCredential.credential` | the same credential parsed, for convenience |

In [11]:
stored = ds.get(f"{ds.identity_api('consumer')}/credentials").json()
membership_entry = next(e for e in stored if "MembershipCredential" in e["verifiableCredential"]["credential"]["type"])

CREDENTIAL_STATES = {100: "INITIAL", 200: "REQUESTING", 300: "REQUESTED", 400: "ISSUING", 500: "ISSUED",
                     600: "REVOKED", 700: "SUSPENDED", 800: "EXPIRED", 900: "NOT_YET_VALID", -100: "ERROR"}

print("credential resource fields:", sorted(membership_entry))
print("state:", membership_entry["state"], "=", CREDENTIAL_STATES[membership_entry["state"]])
credential = membership_entry["verifiableCredential"]["credential"]
print("types:", credential["type"])
print("valid:", credential["issuanceDate"], "->", credential["expirationDate"])
print("claims:")
ds.show(credential["credentialSubject"], max_chars=500)

GET    /consumer/ih/identity/v1alpha/participants/ZGlkOndlYjpjb25zdW1lci1pZGVudGl0eWh1YiUzQTgzODM6YXBpOmRpZA==/credentials  ->  200
credential resource fields: ['createdAt', 'holderId', 'id', 'issuancePolicy', 'issuerId', 'metadata', 'participantContextId', 'reissuancePolicy', 'state', 'timeOfLastStatusUpdate', 'timestamp', 'usage', 'verifiableCredential']
state: 500 = ISSUED
types: ['VerifiableCredential', 'MembershipCredential']
valid: 2026-10-09T12:45:02.330037338Z -> 2036-10-06T12:45:02.330037505Z
claims:
[
  {
    "id": "did:web:consumer-identityhub%3A8383:api:did",
    "name": "consumer",
    "membership": {
      "membershipType": "FullMember",
      "since": "2026-10-09 12:38:14.496"
    },
    "companySegment": "Airlines"
  }
]


### Inside the JWT

A JWT is `header.payload.signature`, each part base64url-encoded. Decoding the first two parts shows how the credential is secured:

- header `alg: ES256`: signed with the issuer's P-256 key; `kid` points to the exact key in the issuer's DID document;
- payload `iss` = issuer DID, `sub` = holder DID, `iat`/`exp` = issued/expires, `vc` = the credential itself.

A verifier fetches the issuer's DID document, takes the key named in `kid`, and checks the signature.

In [12]:
raw_jwt = membership_entry["verifiableCredential"]["rawVc"]
print("header: ", ds.decode_jwt_header(raw_jwt))

claims = ds.decode_jwt_payload(raw_jwt)
print("iss:    ", claims["iss"])
print("sub:    ", claims["sub"])
print("iat/exp:", claims["iat"], "/", claims["exp"])
print("vc type:", claims["vc"]["type"])

header:  {'kid': 'did:web:authority-identityhub%3A8383:api:did#my-key', 'alg': 'ES256'}
iss:     did:web:authority-identityhub%3A8383:api:did
sub:     did:web:consumer-identityhub%3A8383:api:did
iat/exp: 1791549902 / 2106909902
vc type: ['VerifiableCredential', 'MembershipCredential']


### Revocation: the status list

Credentials are valid for 10 years, but the issuer can revoke them. Each credential carries a `credentialStatus` entry pointing to a
**Bitstring Status List**: a credential published by the issuer that holds one bit per issued credential (1 = revoked).
Verifiers download the list and check the bit at `statusListIndex`.

**What:** fetch the list through the ingress (`/authority/is/statuslist/<id>`).  **Returns:** a JWT (`application/vc+jwt`) whose
`encodedList` is the bitstring, gzip-compressed and base64url-encoded (with a leading `u`, the multibase prefix for base64url).

In [13]:
import base64, gzip

status_entry = ds.as_list(credential["credentialStatus"])[0]
print("status entry:", {k: status_entry[k] for k in ("type", "statusPurpose", "statusListIndex")})

list_id = status_entry["statusListCredential"].rsplit("/", 1)[1]
status_list_jwt = ds.get(f"{ds.BASE_URL}/authority/is/statuslist/{list_id}").text
encoded_list = ds.decode_jwt_payload(status_list_jwt)["vc"]["credentialSubject"]["encodedList"]

without_prefix = encoded_list[1:]                                    # drop the multibase "u"
compressed = base64.urlsafe_b64decode(without_prefix + "=" * (-len(without_prefix) % 4))
bits = gzip.decompress(compressed)

index = int(status_entry["statusListIndex"])
revoked = (bits[index // 8] >> (7 - index % 8)) & 1
print(f"list holds {len(bits) * 8} bits; bit {index} = {revoked} -> {'REVOKED' if revoked else 'not revoked'}")

status entry: {'type': 'BitstringStatusListEntry', 'statusPurpose': 'revocation', 'statusListIndex': 56}
GET    /authority/is/statuslist/73f2c5af-22bc-4aff-9b0e-12ad5a1d741e  ->  200
list holds 16384 bits; bit 56 = 0 -> not revoked


## A8 · The issuer's side: issuance processes

**What:** `POST .../issuanceprocesses/query`.  **Returns:** one record per credential request the issuer handled, with the claims it computed.
`DELIVERED` reached the wallet. `ERRORED` gave up after its retries (for instance requests sent with the wrong descriptor field).

In [14]:
from collections import Counter

processes = ds.post(f"{ds.issuer_admin_api()}/issuanceprocesses/query", {}).json()
print(Counter(process["state"] for process in processes))

latest = processes[-1]
print("\nlatest process:", latest["state"], latest["holderPid"])
print("claims the issuer computed from the attestations:")
ds.show(latest["claims"], max_chars=600)

POST   /authority/is/issueradmin/v1alpha/participants/ZGlkOndlYjphdXRob3JpdHktaWRlbnRpdHlodWIlM0E4MzgzOmFwaTpkaWQ=/issuanceprocesses/query  ->  200
Counter({'DELIVERED': 8, 'ERRORED': 7})

latest process: ERRORED did:web:authority-identityhub%3A8383:api:didDomainCredential
claims the issuer computed from the attestations:
{
  "holder_id": "did:web:authority-identityhub%3A8383:api:did",
  "domain": "route",
  "id": "dc40a319-d997-43c4-ae65-fc310ee8d9a0"
}


Next: **02_publishing.ipynb**